# LA Studio Language GPU Worker

Worker Colab này phục vụ **hai API trực tiếp, độc lập với API Gateway**:

- `POST /v1/translations` — M2M100 chạy trên CUDA.
- `POST /v1/chat/completions` — Qwen chat chạy trên CUDA, trả SSE.

Chọn **Runtime → Change runtime type → GPU**, rồi Run all. Chép `LA_STUDIO_LANGUAGE_URL` và `LA_STUDIO_LANGUAGE_TOKEN` vào panel Colab của Translation hoặc LLM Chat. Không thêm `/v1` vào URL. URL/token chỉ tồn tại trong phiên Colab và không liên quan API Gateway.

In [ ]:
import subprocess, sys

def run(command):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
     'fastapi>=0.115,<1', 'uvicorn[standard]>=0.30,<1',
     'transformers>=4.45,<5', 'accelerate>=1,<2',
     'sentencepiece>=0.2,<1', 'safetensors>=0.4,<1'])
print('CUDA worker dependencies installed.')

In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_language_worker.py')
WORKER.write_text(r'''
import json
import os
import secrets
import threading
from typing import Any

import torch
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, M2M100ForConditionalGeneration,
    M2M100Tokenizer, StoppingCriteria, StoppingCriteriaList, TextIteratorStreamer,
)

TOKEN = os.environ['LA_STUDIO_LANGUAGE_API_TOKEN']
CACHE_DIR = os.environ.get('LA_STUDIO_LANGUAGE_CACHE', '/content/la-studio-language-models')
TRANSLATION_MODELS = {
    'm2m100-418m': 'facebook/m2m100_418M',
}
CHAT_MODELS = {
    'qwen2.5-1.5b-instruct': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b-instruct': 'Qwen/Qwen2.5-3B-Instruct',
}

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a Colab GPU runtime; CPU fallback is disabled.')

MODEL_LOCK = threading.Lock()
INFERENCE_SLOTS = threading.BoundedSemaphore(1)
MAX_TRANSLATION_SEGMENTS = 128
MAX_TRANSLATION_CHARS = 50000
MAX_CHAT_MESSAGES = 64
MAX_CHAT_CHARS = 50000
MAX_CHAT_TOKENS = 4096
TRANSLATORS: dict[str, tuple[Any, Any]] = {}
CHATS: dict[str, tuple[Any, Any]] = {}

app = FastAPI(title='LA Studio Language GPU Worker', docs_url=None, redoc_url=None, openapi_url=None)

def authorize(authorization: str = Header(default='')):
    if not secrets.compare_digest(authorization, 'Bearer ' + TOKEN):
        raise HTTPException(status_code=401, detail='invalid or missing bearer token')

def translation_model(model_id: str):
    repo = TRANSLATION_MODELS.get(model_id)
    if not repo:
        raise HTTPException(status_code=400, detail='unsupported translation model: ' + model_id)
    with MODEL_LOCK:
        if model_id not in TRANSLATORS:
            tokenizer = M2M100Tokenizer.from_pretrained(repo, cache_dir=CACHE_DIR)
            model = M2M100ForConditionalGeneration.from_pretrained(
                repo, cache_dir=CACHE_DIR, torch_dtype=torch.float16
            ).to('cuda').eval()
            TRANSLATORS[model_id] = (tokenizer, model)
        return TRANSLATORS[model_id]

def chat_model(model_id: str):
    repo = CHAT_MODELS.get(model_id)
    if not repo:
        raise HTTPException(status_code=400, detail='unsupported chat model: ' + model_id)
    with MODEL_LOCK:
        if model_id not in CHATS:
            tokenizer = AutoTokenizer.from_pretrained(repo, cache_dir=CACHE_DIR)
            model = AutoModelForCausalLM.from_pretrained(
                repo, cache_dir=CACHE_DIR, torch_dtype=torch.float16
            ).to('cuda').eval()
            CHATS[model_id] = (tokenizer, model)
        return CHATS[model_id]

class TranslationSegment(BaseModel):
    id: str = Field(min_length=1, max_length=128)
    sourceText: str = Field(min_length=1, max_length=5000)

class TranslationRequest(BaseModel):
    model: str = 'm2m100-418m'
    source_language: str = Field(min_length=2, max_length=8)
    target_language: str = Field(min_length=2, max_length=8)
    segments: list[TranslationSegment]

class ChatMessage(BaseModel):
    role: str = Field(pattern='^(system|user|assistant)$')
    content: str = Field(min_length=1, max_length=8000)

class ChatRequest(BaseModel):
    model: str = 'qwen2.5-3b-instruct'
    messages: list[ChatMessage]
    stream: bool = True
    max_tokens: int = Field(default=1024, ge=1, le=MAX_CHAT_TOKENS)
    temperature: float = Field(default=0.7, ge=0.01, le=2.0)
    top_p: float = Field(default=0.8, ge=0.01, le=1.0)

@app.get('/health')
def health(_: None = Depends(authorize)):
    return {'ready': True, 'device': 'cuda', 'translation_loaded': list(TRANSLATORS), 'chat_loaded': list(CHATS)}

@app.get('/v1/capabilities')
def capabilities(_: None = Depends(authorize)):
    return {'contract_version': 1, 'device': 'cuda', 'translation': [{'id': key, 'loaded': key in TRANSLATORS} for key in TRANSLATION_MODELS],
            'chat': [{'id': key, 'loaded': key in CHATS} for key in CHAT_MODELS]}

@app.post('/v1/translations')
def translate(request: TranslationRequest, _: None = Depends(authorize)):
    if not request.segments:
        raise HTTPException(status_code=400, detail='segments must not be empty')
    texts = [item.sourceText for item in request.segments]
    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(len(text) for text in texts) > MAX_TRANSLATION_CHARS:
        raise HTTPException(status_code=413, detail='translation request is too large')
    if any(not text.strip() for text in texts):
        raise HTTPException(status_code=400, detail='each segment needs sourceText')
    if not INFERENCE_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='worker is busy; retry shortly')
    try:
        source = request.source_language.lower()
        target = request.target_language.lower()
        try:
            tokenizer, model = translation_model(request.model)
            tokenizer.src_lang = source
            forced_bos_token_id = tokenizer.get_lang_id(target)
        except KeyError:
            raise HTTPException(status_code=400, detail='unsupported M2M100 language code')
        with torch.inference_mode():
            encoded = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
            generated = model.generate(**encoded, forced_bos_token_id=forced_bos_token_id, max_new_tokens=512)
            output = tokenizer.batch_decode(generated, skip_special_tokens=True)
        return {'patches': [{'id': item.id, 'targetText': text.strip(), 'state': 'translated'}
                            for item, text in zip(request.segments, output)]}
    finally:
        INFERENCE_SLOTS.release()

class DisconnectStop(StoppingCriteria):
    def __init__(self, cancelled: threading.Event):
        self.cancelled = cancelled
    def __call__(self, input_ids, scores, **kwargs):
        return self.cancelled.is_set()

@app.post('/v1/chat/completions')
async def chat(request: ChatRequest, http_request: Request, _: None = Depends(authorize)):
    if not request.stream:
        raise HTTPException(status_code=400, detail='this direct worker requires stream=true')
    if not request.messages:
        raise HTTPException(status_code=400, detail='messages must not be empty')
    if len(request.messages) > MAX_CHAT_MESSAGES or sum(len(item.content) for item in request.messages) > MAX_CHAT_CHARS:
        raise HTTPException(status_code=413, detail='chat request is too large')
    if not INFERENCE_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='worker is busy; retry shortly')
    try:
        tokenizer, model = chat_model(request.model)
        messages = [{'role': item.role, 'content': item.content} for item in request.messages]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        cancelled = threading.Event()
        generation = threading.Thread(target=model.generate, kwargs={
            **inputs, 'streamer': streamer, 'max_new_tokens': request.max_tokens,
            'do_sample': True, 'temperature': request.temperature, 'top_p': request.top_p,
            'pad_token_id': tokenizer.eos_token_id,
            'stopping_criteria': StoppingCriteriaList([DisconnectStop(cancelled)]),
        }, daemon=True)
        generation.start()
    except Exception:
        INFERENCE_SLOTS.release()
        raise
    async def events():
        try:
            for token in streamer:
                if await http_request.is_disconnected():
                    cancelled.set()
                    break
                payload = {'choices': [{'delta': {'content': token}}]}
                yield 'data: ' + json.dumps(payload, ensure_ascii=False) + '\n\n'
            yield 'data: [DONE]\n\n'
        finally:
            cancelled.set()
            INFERENCE_SLOTS.release()
    return StreamingResponse(events(), media_type='text/event-stream', headers={'Cache-Control': 'no-store'})
''', encoding='utf-8')
print('Worker source prepared:', WORKER)

In [ ]:
import json, os, secrets, subprocess, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update({'LA_STUDIO_LANGUAGE_API_TOKEN': TOKEN, 'LA_STUDIO_LANGUAGE_CACHE': '/content/la-studio-language-models'})
log_path = '/content/la-studio-language-worker.log'
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_language_worker:app',
                           '--host', '127.0.0.1', '--port', '3943'],
                          cwd='/content', env=env, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        health_request = urllib.request.Request('http://127.0.0.1:3943/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(health_request, timeout=5) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Language worker did not become CUDA-ready. Log tail:\n' + Path(log_path).read_text(errors='replace')[-4000:])

subprocess.run(['wget', '-q', '-O', '/content/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
subprocess.run(['dpkg', '-i', '/content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3943', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    if 'https://' in line and 'trycloudflare.com' in line:
        public_url = line[line.find('https://'):].split()[0]
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found.')

print('\nLA_STUDIO_LANGUAGE_URL=' + public_url)
print('LA_STUDIO_LANGUAGE_TOKEN=' + TOKEN)
print('TRANSLATION_MODEL=m2m100-418m')
print('CHAT_MODEL=qwen2.5-3b-instruct')
print('Paste the URL and token into a Colab GPU panel. Do not add /v1 to the URL.')